01: Get all repositories under "Android" topic

In [ ]:
from datetime import timedelta
import requests
import json

github_key1 = "<your_github_key1>"
github_key2 = "<your_github_key1>"
github_key3 = "<your_github_key1>"

breakp_field='created'
query = 'https://api.github.com/search/repositories?q=topic:android+stars:>0'
headers = {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"}
num_pages = 10

def getSearchResultsCount(query,qualifier,page):
    while True:
        response = requests.get(query+"+"+breakp_field+":"+qualifier+"&per_page=100&page="+str(page))

        # Check if response was successful (status code 200)
        if response.status_code == 200:
            # Convert response to JSON format
            response_json = response.json()
            # Check if total_count is greater than 1000
            return response_json['total_count'],response_json['items']
        else:
            print(response.status_code)

def iterateOverPages(qualifier,json_file):
    # Iterate over each page of results

    for p in range(1, num_pages + 1):
        _,items=getSearchResultsCount(query,qualifier,p)
        for repo in items:
            json.dump(repo, json_file)
            json_file.write(',\n')

def segment_time_period(start_time, end_time, min_count=750, max_count=1000):

    delta = timedelta(hours=720*5) # change this to adjust the length of each time segment
    #time_segments = []
    curr_time = start_time
    start_segment = curr_time
    end_segment = start_segment+delta
    with open('../Data/android_repos.json', mode='a') as json_file:
        iterateOverPages("<2017-09-01",json_file)
        tries=0
        while start_segment < end_time:
            # determine the start and end time for the current time segment    
            
            if end_segment > end_time:
                end_segment = end_time

            # count the number of posts in the current time segment
            count,results=getSearchResultsCount(query,start_segment.strftime('%Y-%m-%d')+".."+end_segment.strftime('%Y-%m-%d'),1)


            # if the count is within the desired range, add the time segment to the list
            if min_count <= count <= max_count:
                print(start_segment.strftime('%Y-%m-%d')+".."+end_segment.strftime('%Y-%m-%d')+" count: "+str(count))
                iterateOverPages(start_segment.strftime('%Y-%m-%d')+".."+end_segment.strftime('%Y-%m-%d'),json_file)
                print('done')
                tries=0
                delta = timedelta(hours=720*5)
                start_segment=end_segment
                end_segment=end_segment+ delta
            else:
                # if the count is outside the desired range, adjust the length of the time segment
                # by increasing or decreasing the delta value
                
                if count < min_count:
                    if end_segment >= end_time or tries>3:
                        print(start_segment.strftime('%Y-%m-%d')+".."+end_segment.strftime('%Y-%m-%d')+" count: "+str(count))
                        iterateOverPages(start_segment.strftime('%Y-%m-%d')+".."+end_segment.strftime('%Y-%m-%d'),json_file)
                        print('done')
                        tries=0
                        delta = timedelta(hours=720*5)
                        start_segment=end_segment
                        end_segment=end_segment+ delta
                        if end_segment >= end_time:
                            break
                    else:
                        delta *= 1.1
                        tries+=1
                        end_segment = end_segment + delta
                        print(start_segment.strftime('%Y-%m-%d')+".."+end_segment.strftime('%Y-%m-%d')+" delta: "+str(delta)+ " count:"+str(count))
                else:
                    delta *= 0.9
                    tries+=1
                    end_segment = start_segment + delta
                    print(start_segment.strftime('%Y-%m-%d')+".."+end_segment.strftime('%Y-%m-%d')+" end_segment: "+end_segment.strftime('%Y-%m-%d')+ " count:"+str(count))
    

02: Check if the repo is valid android application and collect manifest info

In [ ]:
import json
import time
from langdetect import detect
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from google_play_scraper import app, exceptions
import base64
import json
import os
import re
import shutil

# Define a list of authorization headers
authorization_headers = [
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key2}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key3}"}
]

def makeRequest(repo_url):
    tries_flag = 0
    headers = {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"}
    failed_request_retrials = 0
    max_failed_request_retrials = 1
    while True:
        response = requests.get(repo_url, headers=headers)
        if response.status_code == 200:
            return response.json()
        elif response.status_code in [403, 429] and ('message' in response.json() and "rate limit" in response.json()["message"].lower()):
            tries_flag += 1
            if tries_flag > len(authorization_headers):
                reset_timestamp = int(response.headers['X-RateLimit-Reset'])
                current_timestamp = int(time.time())
                wait_time = reset_timestamp - current_timestamp
                print(f"Rate limit exceeded. Waiting for {wait_time} seconds...")
                time.sleep(wait_time + 1)
                tries_flag = 0
            else:
                # Cycle through the authorization headers
                headers = authorization_headers[tries_flag - 1]
        elif response.status_code in[422, 401]:
            return None
        else:
            print(f"Request failed with status code {response.status_code}. Retrying... " + repo_url)
            if failed_request_retrials == max_failed_request_retrials:
                return None
            failed_request_retrials += 1
            time.sleep(1)

df = pd.read_csv("../Data/our_original_repos.csv")
output_file_exists = os.path.isfile('android_repos_manifest_paths.csv')
with open('../Data/android_repos_manifest_paths.csv', mode='a', encoding='utf-8') as csv_file:
    if not output_file_exists:
        csv_file.write('repo,num_apps,manifest_count,package_count,manifest_paths,package_ids\n')
    for index, row in df.iterrows():
        url = "https://api.github.com/search/code?q=<activity+filename:AndroidManifest.xml+repo:"
        response=makeRequest(url+row['repo'])
        androidmanifest_count=response["total_count"] if response else 0

        if androidmanifest_count > 0:
            packages=[]
            manifest_paths=[]
            for item in response["items"]:
                manifest_url = item["url"]
                manifest_path = item["path"]
                manifest_paths.append(manifest_path)

                response = makeRequest(manifest_url)

                manifest_xml = base64.b64decode(response['content']).decode('utf-8')
                package_value = ""
                try:
                    root = ET.fromstring(manifest_xml)
                    package_value = root.attrib.get("package")
                except Exception as ex:
                    print(ex)
                if package_value is not None and package_value != "":
                    packages.append(package_value)

            csv_file.write(row['repo']+','+str(androidmanifest_count)+','+str(len(set(manifest_paths)))+','+str(len(set(packages)))+',['+','.join(set(manifest_paths))+']'+',['+','.join(set(packages))+']\n')
            csv_file.flush()


03: Crawl all FDroid applications

In [ ]:
import requests
import json
from bs4 import BeautifulSoup #pip install requests beautifulsoup4

# Function to extract app data from a given app URL
def extract_app_data(app_url, category_name):
    print(app_url)
    app_response = requests.get(app_url)
    app_soup = BeautifulSoup(app_response.content, 'html.parser')
    
    app_title = app_soup.find('h3', class_='package-name').text.strip()
    print(app_title)
    app_description = app_soup.find('div', class_='package-summary').text.strip()
    
    has_source_code, source_code_url = check_source_code_link(app_soup)
    
    app_dict = {
        'Title': app_title,
        'Description': app_description,
        'Category': category_name,
        'Has Source Code': has_source_code,
        'Source Code URL': source_code_url,
        'URL': app_url
    }
    
    # Open the JSON file in append mode and write the app data
    with open('../Data/fdroid_apps_data.json', 'a') as json_file:
        json.dump(app_dict, json_file)
        json_file.write(',\n')

# Function to check for source code links in app page
def check_source_code_link(app_soup):
    # Check for "source code" link
    source_code_link = app_soup.find('a', string='Source Code')
    if source_code_link:
        return True, source_code_link['href']
    
    # Check for open-source platform links
    platforms = ['github', 'gitlab', 'bitbucket']
    for platform in platforms:
        platform_link = app_soup.find('a', href=lambda href: href and platform in href.lower())
        if platform_link:
            # Check if it is a repository link
            if 'issues' not in platform_link['href'] and 'pulls' not in platform_link['href']:
                return True, platform_link['href']
    
    return False, None

# Function to read page links in category page
def read_page_apps(page_url,category_name):

    page_response = requests.get(page_url)
    page_soup = BeautifulSoup(page_response.content, 'html.parser')

    # Find all app links on the category page
    page_app_list = page_soup.find('div', id='package-list')
    page_app_links = page_app_list.find_all('a', class_='package-header')
    # Iterate over each app link in the category page
    for page_app_link in page_app_links:
        page_app_url = base_url + page_app_link['href']        
        extract_app_data(page_app_url, category_name)
    return page_soup

# URL of the F-Droid website
base_url = 'https://f-droid.org'

# Send a GET request to the home page
response = requests.get(base_url + '/en/packages/')

# Create a BeautifulSoup object to parse the HTML content
soup = BeautifulSoup(response.content, 'html.parser')

# Find all category divs on the home page
category_parent = soup.find('div', class_='post-content')
category_divs = category_parent.find_all('h3')

print("Categories No.: "+str(len(category_divs)))
# List to store the extracted app data
count=1
with open('../Data/fdroid_apps_data.json', 'w') as json_file:
    json_file.write('[')

# Iterate over each category div
for category_div in category_divs:
    # Extract category name
    category_name = category_div.text.strip()
    print("Start crawling category `"+category_name+ "` ...")

    show_all_url = base_url + "/en/categories/"+category_name.lower().replace(" ","").replace("&","-")+"/"
    category_response = requests.get(show_all_url)
    category_soup = BeautifulSoup(category_response.content, 'html.parser')
    #read first page
    print("Start crawling page "+show_all_url+" ...")
    read_page_apps(show_all_url,category_name)

    while True:
        # Find all navigation pages links on the category page
        next_page_link = category_soup.find('li', class_='next')

        if next_page_link.a:
            page_url = base_url + next_page_link.a['href']
            print("Start crawling page "+page_url+" ...")
            category_soup=read_page_apps(page_url,category_name)
        else: 
            break
# Save the app data to a JSON file
with open('../Data/fdroid_apps_data.json', 'a') as json_file:
    json_file.write(']')

print('Data saved to apps_data.json')

04: Check if FDroid application has valid repo and is valid android application

In [ ]:
import json
import time
from langdetect import detect
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from google_play_scraper import app, exceptions
import base64
import json

url = f"https://api.github.com/search/code?q=activity+filename:AndroidManifest.xml+repo:"

# Define a list of authorization headers
authorization_headers = [
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key2}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key3}"}
]

def makeRequest(url):
    tries_flag = 0
    headers=   {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"}
    while True:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 403 and ('message' in response.json() and "rate limit" in response.json()["message"].lower()):
            tries_flag += 1
            if tries_flag > len(authorization_headers):
                reset_timestamp = int(response.headers['X-RateLimit-Reset'])
                current_timestamp = int(time.time())
                wait_time = reset_timestamp - current_timestamp
                print(f"Rate limit exceeded. Waiting for {wait_time} seconds...")
                time.sleep(wait_time + 1)
                tries_flag = 0
            else:
                # Cycle through the authorization headers
                headers = authorization_headers[tries_flag - 1]
        elif response.status_code == 422:
            return {"total_count": 0}
        elif response.status_code == 404:
            return {"total_count": 0}
        else:
            print(f"Request failed with status code {response.status_code}. Retrying... " + url)
            time.sleep(1)

df=pd.read_json('fdroid_apps_data.json')
df = df[df['owner'].notnull()]

for index, obj in df.iterrows():
    app_repo=obj.copy()
    response=makeRequest(url+str(obj["owner"])+"/"+str(obj["name"]))
    # Check if the file was found
    androidmanifest_count=response["total_count"]
    app_repo['androidmanifest_count']=androidmanifest_count
  
    packages=[]
    package_counts=0
    google_play_data=[]
    for item in response["items"]:
        # Get the URL of the  search result
        manifest_url = item["url"]
        manifest_path=item["path"]
        response =makeRequest(manifest_url)
        if 'content' in response:
            # Retrieve the content of the AndroidManifest.xml file
            manifest_xml = base64.b64decode(response['content']).decode('utf-8')
            package_value=""
            # Parse the XML file and retrieve the package value
            try:
                root = ET.fromstring(manifest_xml)
                package_value = root.attrib.get("package")
            except:
                google_play_data.append("") 
            if package_value is not None and package_value!="":
                package_counts+=1
                packages.append(package_value)
                try:
                    result = app(package_value.encode('utf-8'),lang=detect(package_value))
                    if result:
                        google_play_data.append(result)
                    else:
                        google_play_data.append("")
                except exceptions.NotFoundError:
                        google_play_data.append("")
        else:
            google_play_data.append("")
    app_repo['package_counts']=package_counts
    app_repo['packages']=packages
    app_repo['google_play_data']=google_play_data
    with open('../Data/fdroid_repos_packages.json', mode='a', encoding='utf-8') as json_file:
        json.dump(app_repo.to_dict(), json_file)
        json_file.write(',\n')
        print('done')

05: Merge datasets from github and FDroid sources and remove duplicates

In [ ]:
# merge datasets into one
import csv
import json

# Read FDroid dataset
with open('../Data/fdroid_repos_packages.json', 'r') as fdroid_file:
    fdroid_data = json.load(fdroid_file)

# Read GitHub dataset
with open('../Data/android_repos_ci.json', 'r') as github_file:
    github_data = json.load(github_file)

# Create a merged dataset
merged_data = []

# Store FDroid app names for quick lookup
fdroid_app_names = set(fdroid_app['name'] for fdroid_app in fdroid_data)

# Process FDroid dataset
for fdroid_app in fdroid_data:
    app_name = fdroid_app['owner']+"/"+fdroid_app['name']

    # Skip FDroid apps that exist in GitHub dataset
    if app_name in fdroid_app_names.intersection(set(github_app['owner']+"/"+github_app['name'] for github_app in github_data)):
        continue

    app_title =  fdroid_app['owner']+"/"+fdroid_app['name']
    fdroid_title=fdroid_app['Title']
    fdroid_category = fdroid_app['Category']
    is_fdroid = 1
    repo_url = fdroid_app['Source Code URL']
    language = fdroid_app['language']
    repo_source = 'FDroid'
    num_ci_tools = fdroid_app['num_ci_tools']
    ci_tools=fdroid_app['ci_tools']
    repo_forks = fdroid_app['num_forks']
    repo_stars = fdroid_app['num_stars']
    issues_count=fdroid_app['open_issues_count']
    num_watchers=fdroid_app['num_watchers']
    package_counts=fdroid_app['package_counts']
    packages=fdroid_app['packages']
    homepage=fdroid_app['homepage']
    size=fdroid_app['size']
    repo_last_activity = fdroid_app['last_updated']
    repo_first_activity = fdroid_app['creation_date']
    app_license=fdroid_app['license']

    # Create a merged app object
    merged_app = {
        'App_Name': app_name,
    #    'App_Title':app_title,
        'FDroid_Name': fdroid_title,
        'FDroid_Category': fdroid_category,
        'Is_FDroid': is_fdroid,
        'Repo_URL': repo_url,
        'Language':language,
        'Repo_Source': repo_source,
        'Num_CI_Tools': num_ci_tools,
        'CI_Tools':ci_tools,
        'Repo_Forks': repo_forks,
        'Repo_Stars': repo_stars,
        'Issues_Count':issues_count,
        'Watchers_Count':num_watchers,
        'Package_Counts':package_counts,
        'Packages':packages,
        'Homepage':homepage,
        'Size':size,
        'Repo_Last_Activity': repo_last_activity,
        'Repo_First_Activity': repo_first_activity,
        'License':app_license
    }

    merged_data.append(merged_app)

# Process GitHub dataset
for github_app in github_data:
    app_name = github_app['owner']+"/"+github_app['name']

    # Skip GitHub apps that exist in FDroid dataset
    if app_name in fdroid_app_names:
        continue

    app_title =  github_app['owner']+"/"+github_app['name']
    fdroid_title='-'
    fdroid_category = '-'
    is_fdroid = 0
    repo_url = "https://github.com/"+github_app['owner']+"/"+github_app['name']
    language = github_app['language']
    repo_source = 'Github'
    num_ci_tools = github_app['num_ci_tools']
    ci_tools=github_app['ci_tools']
    repo_forks = github_app['num_forks']
    repo_stars = github_app['num_stars']
    issues_count=github_app['open_issues_count']
    num_watchers=github_app['num_watchers']
    package_counts=github_app['package_counts']
    packages=github_app['packages']
    homepage=github_app['homepage']
    size=github_app['size']
    repo_last_activity = github_app['last_updated']
    repo_first_activity = github_app['creation_date']
    app_license=github_app['license']

    # Create a merged app object
    merged_app = {
        'App_Name': app_name,
     #   'App_Title':app_title,
        'FDroid_Name': fdroid_title,
        'FDroid_Category': fdroid_category,
        'Is_FDroid': is_fdroid,
        'Repo_URL': repo_url,
        'Language':language,
        'Repo_Source': repo_source,
        'Num_CI_Tools': num_ci_tools,
        'CI_Tools':ci_tools,
        'Repo_Forks': repo_forks,
        'Repo_Stars': repo_stars,
        'Issues_Count':issues_count,
        'Watchers_Count':num_watchers,
        'Package_Counts':package_counts,
        'Packages':packages,
        'Homepage':homepage,
        'Size':size,
        'Repo_Last_Activity': repo_last_activity,
        'Repo_First_Activity': repo_first_activity,
        'License':app_license
    }

    merged_data.append(merged_app)

# Save the merged dataset to a CSV file
csv_filename = 'new_merged_dataset.csv'
csv_fields = list(merged_data[0].keys())

with open(csv_filename, 'w', newline='') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
    writer.writeheader()
    writer.writerows(merged_data)

06: Check if apps use CI/CD services

In [ ]:
import re
import requests
import json
import csv
import time
from urllib.parse import urlparse
import pandas as pd
authorization_headers = [
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key2}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key3}"}
]

def makeRequest(url):
    tries_flag = 0
    headers=   {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"}
    while True:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 403 and ('message' in response.json() and "rate limit" in response.json()["message"].lower()):
            tries_flag += 1
            if tries_flag > len(authorization_headers):
                reset_timestamp = int(response.headers['X-RateLimit-Reset'])
                current_timestamp = int(time.time())
                wait_time = reset_timestamp - current_timestamp
                print(f"Rate limit exceeded. Waiting for {wait_time} seconds...")
                time.sleep(wait_time + 1)
                tries_flag = 0
            else:
                # Cycle through the authorization headers
                headers = authorization_headers[tries_flag - 1]
        elif response.status_code == 422:
            return {"total_count": 0}
        elif response.status_code == 404:
            return {"total_count": 0}
        else:
            print(f"Request failed with status code {response.status_code}. Retrying... " + url)
            time.sleep(1)

def check_ci(repo_url):
    ci_tools = []
    query = f"extension:yml extension:yaml repo:{repo_url}"
    api_url = f"https://api.github.com/search/code?q={query}&per_page=100&page=1"
    response = makeRequest(api_url)

    if response["total_count"]>0:
        results = response["items"]

        # dictionary of file name patterns to CI tool names
        ci_file_patterns = {
            r'.travis\.yml$': 'Travis CI',
            r'(.appveyor\.yml|appveyor\.yml)$': 'AppVeyor',
            r'(\.circleci/config\.yml|circle\.yml)$': 'CircleCI',
            r'azure-pipelines\.yml$': 'Azure Pipelines',
            r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
            r'bitbucket-pipelines\.yml$':'Bitbucket',
            r'.gitlab-ci\.yml$': 'GitLab',
            r'jenkins': 'Jenkins',
            r'bitrise\.yml$': 'Bitrise',
            r'bamboo\.yml$': 'Bamboo',
            r'codeship-services\.yml$': 'Codeship',
            r'.gocd\.yaml$': 'GoCD',
            r'wercker\.yaml$': 'Wercker',
            r'semaphore\.yml$': 'Semaphore',
            r'codemagic\.yaml$': 'Nevercode',
        }

        for result in results:
            path = result["path"].lower()

            for pattern, ci_tool_name in ci_file_patterns.items():
                if re.match(pattern, path):
                    ci_tools.append(ci_tool_name)
                    break
    
    return ci_tools

android_apps=pd.read_csv('../Data/new_merged_dataset.csv')
with open('../Data/android_apps_ci.json', 'a') as f:
    f.write('[')
count=0
for index,app in android_apps.iterrows():
    count+=1
    ci_tools =list(set(check_ci(app['App_Name']))) 
    ci_tools_count=len(ci_tools)
    app['num_ci_tools']=ci_tools_count
    app['ci_tools']=ci_tools
    with open('../Data/android_apps_ci.json', 'a') as f:
        json.dump(app.to_dict(), f)
        f.write(']') if index==len(android_apps)-1 else f.write(',\n')
        print(str(count)+": "+app['App_Name']+".. done. ci count:"+str(ci_tools_count))

Collect android repos About section

In [ ]:
repos = pd.read_csv("../Data/our_original_repos.csv")
output_file_exists = os.path.isfile('android_repos_descriptions.csv')
with open('../Data/android_repos_descriptions.csv', mode='a', encoding='utf-8') as csv_file:
    if not output_file_exists:
        csv_file.write('repo,description\n')
    for index, row in repos.iterrows():
        repo = row['repo']
        description = ''

        url = "https://api.github.com/repos/"
        response=makeRequest(url+repo)

        if response:
            description = response.get("description", "")
            description = description.replace('\r\n', ' ').replace('\n', ' ').replace('"', "'") if description else ''
        
        csv_file.write(row['repo']+',"'+description+'"\n')
        csv_file.flush()

Collect readme files

In [ ]:
repos = pd.read_csv("../Data/our_original_repos.csv")

for index, row in repos.iterrows():
    repo = row['repo']

    url = f'https://api.github.com/repos/{repo}/contents/README.md'
    response=makeRequest(url)
    
    with open('../Data/ReadmeFiles/' + row['repo'].replace('/', '@') + '.md', mode='w', encoding='utf-8') as readme_file:
        if response:
            readme_content = base64.b64decode(response['content']).decode('utf-8')
            readme_file.write(readme_content)
        readme_file.close()


07: Remove toy projects

In [ ]:
android_repos_manifest_paths = pd.read_csv('../Data/android_repos_manifest_paths.csv')
android_repos_descriptions = pd.read_csv('../Data/android_repos_descriptions.csv')
changes_data_with_categories = pd.read_csv('../Data/changes_data_with_categories.csv')

# Merge manifest data with repos descriptions data
all_projects_manifest_and_description_data = pd.merge(android_repos_manifest_paths, android_repos_descriptions, on='Repository')

keywords_indicating_non_real_apps = ['example', 'sample', 'demo', 'test', 'debug', 'presentation', 'module', 'components', 'lib', 'library', 'sdk', 'utils', 'utility', 'plugin', 'widget', 'playground', 'framework', 'architecture', 'skeleton', 'collection', 'starting point', 'protocol', 'benchmark', 'hackathon', 'classroom', 'course', 'exercise', 'assignment', 'homework', 'assessment', 'interview', 'asset', 'template', 'catalog', 'tutorial']

# Exclude projects having the above specific keywords as part of their manifest path
all_projects_manifest_and_description_data = all_projects_manifest_and_description_data[~all_projects_manifest_and_description_data['manifest_path'].apply(
    lambda path: any(
        keyword.lower() in path.lower() for keyword in keywords_indicating_non_real_apps
    ))]

keywords_indicating_non_real_apps.append('tool') # Some apps could be real but their manifest is in a folder called tool or tools

# Exclude projects having the above specific keywords as part of their repository name
all_projects_manifest_and_description_data = all_projects_manifest_and_description_data[~all_projects_manifest_and_description_data['Repository'].apply(
    lambda repo: any(
        keyword.lower() in repo.lower() for keyword in keywords_indicating_non_real_apps
    ))]

# Exclude projects having the above specific keywords as part of their description (About section)
not_preceded_by_words = ['using', 'with']
all_projects_manifest_and_description_data = all_projects_manifest_and_description_data[~all_projects_manifest_and_description_data['description'].apply(
    lambda description: any(
        bool(re.search(r'(?<!\S)' + re.escape(keyword) + r'(?!\S)', str(description), re.IGNORECASE)) and not any(
            bool(re.search(r'(?<!\S)' + re.escape(word) + r'\s+(\S+\s+){0,4}' + re.escape(keyword) + r'(?!\S)', str(description), re.IGNORECASE))
            for word in not_preceded_by_words
        )
        for keyword in keywords_indicating_non_real_apps
    ))]

# Exclude projects having the above specific keywords as part of their README.md
preceded_by_words = ['this', 'is a', 'is an', 'our', 'my']
all_projects_manifest_and_description_data = all_projects_manifest_and_description_data[~all_projects_manifest_and_description_data['Repository'].apply(
    lambda repo: any(
        bool(re.search(r'(?<!\S)' + re.escape(keyword) + r'(?!\S)', str(content), re.IGNORECASE)) and any(
            bool(re.search(r'(?<!\S)' + re.escape(word) + r'\s+(\S+\s+){0,4}' + re.escape(keyword) + r'(?!\S)', str(content), re.IGNORECASE))
            for word in preceded_by_words
        )
        for keyword in keywords_indicating_non_real_apps
            if (content := open('../Data/ReadmeFiles/' + repo.replace('/', '@') + '.md', 'r', encoding='utf-8').read())
    ))]

# Merge manifest and description data with the original repos data
merged_real_apps_df = pd.merge(changes_data_with_categories, all_projects_manifest_and_description_data, on='Repository')
merged_real_apps_df.to_csv('../Data/latest_final_changes_data_with_categories.csv', index=False)

pr_and_issues_commits = merged_real_apps_df.drop_duplicates(subset=['Commit Message'])
pr_and_issues_commits = pr_and_issues_commits[pr_and_issues_commits['Commit Message'].str.contains('#') & ~pr_and_issues_commits['Commit Message'].str.contains('#NAME?|#noref', na=False)].copy()
pr_and_issues_commits['is_issue'] = pr_and_issues_commits['Commit Message'].str.contains(r'Fixes #\d+', case=False)
pr_and_issues_commits['PR'] = pr_and_issues_commits['Commit Message'].str.extract(r'#(\d+)')
pr_and_issues_commits['Pull URL'] = 'https://github.com/' + pr_and_issues_commits['Repository'] + pr_and_issues_commits.apply(lambda x: '/issues/' + str(x['PR']) if x['is_issue'] else '/pull/' + str(x['PR']), axis=1).astype(str)
pr_and_issues_commits.to_csv('../Data/pr_and_issues_commits.csv', index=False)

unique_repos = pd.DataFrame(merged_real_apps_df['Repository'].unique(), columns=['Repository'])
unique_repos.to_csv('../Data/our_latest_repos.csv', index=False)

excluded_repos = android_repos_manifest_paths[~android_repos_manifest_paths['Repository'].isin(unique_repos['Repository'])]
excluded_repos = pd.DataFrame(excluded_repos['Repository'].unique(), columns=['Repository'])
excluded_repos.to_csv('../Data/excluded_repos.csv', index=False)


Update Dataset

In [ ]:

our_latest_repos = pd.read_csv('../Data/our_latest_repos.csv')
changes_data_with_categories_json = pd.read_json('../Data/changes_data_with_categories.json')
latest_final_changes_data_with_categories_json = changes_data_with_categories_json[changes_data_with_categories_json['Repository'].isin(our_latest_repos['Repository'])]
latest_final_changes_data_with_categories_json.to_json('../Data/changes_data_with_categories.json', orient='records', indent=4)

latest_final_changes_data_with_categories_csv = pd.read_csv('../Data/changes_data_with_categories.csv')
latest_final_changes_data_with_categories_csv = latest_final_changes_data_with_categories_csv[latest_final_changes_data_with_categories_csv['Repository'].isin(our_latest_repos['Repository'])]
latest_final_changes_data_with_categories_csv.to_csv('../Data/changes_data_with_categories.csv', index=False)

full_dataset = pd.read_csv('../Data/full_dataset.csv')
full_dataset = full_dataset[full_dataset['Repository'].isin(our_latest_repos['Repository'])]
full_dataset.to_csv('../Data/full_dataset.csv', index=False)
full_dataset.to_json('../Data/full_dataset.json', orient='records', indent=4)

07: Collect CI/CD configuration files

In [ ]:
import re
import requests
import time
import os
import pandas as pd

df=pd.read_json('../Data/android_apps_ci.json')
df=df[df.Num_CI_Tools>0]

authorization_headers = [
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key2}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key3}"}
]

def makeRequest(url):
    tries_flag = 0
    headers=   {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"}
    while True:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 403 and ('message' in response.json() and "rate limit" in response.json()["message"].lower()):
            tries_flag += 1
            if tries_flag > len(authorization_headers):
                reset_timestamp = int(response.headers['X-RateLimit-Reset'])
                current_timestamp = int(time.time())
                wait_time = reset_timestamp - current_timestamp
                print(f"Rate limit exceeded. Waiting for {wait_time} seconds...")
                time.sleep(wait_time + 1)
                tries_flag = 0
            else:
                # Cycle through the authorization headers
                headers = authorization_headers[tries_flag - 1]
        elif response.status_code == 422:
            return {"total_count": 0}
        else:
            print(f"Request failed with status code {response.status_code}. Retrying... " + url)
            time.sleep(1)

def download_save_file(repo_name,item,repo_output_folder):
    download_url ="https://raw.githubusercontent.com/"+repo_name+"/"+"/".join(item['html_url'].split('/')[6:])
    file_name = os.path.join(repo_output_folder, item['name'])
    with requests.get(download_url) as yaml_response:
        with open(file_name, 'wb') as f:
            f.write(yaml_response.content)

def download_yaml_files(repo_list, output_folder):
    base_url = 'https://api.github.com/repos/{}/contents'
    count=0
    for repo_name in repo_list:
        count+=1
        print(str(count)+" - "+repo_name)
        repo_output_folder = os.path.join(output_folder, repo_name.replace('/', '_'))
        os.makedirs(repo_output_folder, exist_ok=True)

        # Get the contents of the repository
        url = base_url.format(repo_name)
        query = f"extension:yml extension:yaml repo:{repo_name}"
        api_url = f"https://api.github.com/search/code?q={query}&per_page=100&page=1"
        response = makeRequest(api_url)
        results = response["items"]
        ci_file_patterns = {
            r'.travis\.yml$': 'Travis CI',
            r'(.appveyor\.yml|appveyor\.yml)$': 'AppVeyor',
            r'(\.circleci/config\.yml|circle\.yml)$': 'CircleCI',
            r'azure-pipelines\.yml$': 'Azure Pipelines',
            r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
            r'bitbucket-pipelines\.yml$':'Bitbucket',
            r'.gitlab-ci\.yml$': 'GitLab',
            r'JenkinsFile\.yml$': 'Jenkins',
        }

        for result in results:
            path = result["path"].lower()
            for pattern, ci_tool_name in ci_file_patterns.items():
                if re.match(pattern, path):
                    download_save_file(repo_name,result,repo_output_folder)
                    break

# Create the output folder if it doesn't exist
os.makedirs('../Data/ConfigFiles', exist_ok=True)

# Call the function to download YAML files
download_yaml_files(df['App_Name'].to_list(), '../Data/ConfigFiles')

print("YAML files downloaded and organized by repository successfully!")

08: Parse yml files to get directives and their values

In [ ]:
# Regenerate config items
import os
import yaml #!pip install pyyaml
import csv
from yaml import BaseLoader, Dumper
import pandas as pd
import json

def flatten_dict_or_list(data, parent_key='', sep='.'):
    items = {}
    if isinstance(data, dict):
        for k, v in data.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, (dict, list)):
                items.update(flatten_dict_or_list(v, new_key, sep))
            else:
                items[new_key] = v
    elif isinstance(data, list):
        for idx, item in enumerate(data):
            new_key = f"{parent_key}{sep}{idx}" if parent_key else str(idx)
            if isinstance(item, (dict, list)):
                items.update(flatten_dict_or_list(item, new_key, sep))
            else:
                items[new_key] = item
    return items

def parse_yml_files(repo_file):
    repository = repo_file['Repository']
    ci_file_path = repo_file['CI File Path']
    tool_name = repo_file['CI Tool']
    local_path = repo_file['Local File Path']

    yml_path = local_path
    print(f"Parsing: {yml_path}")
    
    parsed_data = []

    with open(folder_path+yml_path, "r") as yml_file:
        try:
            yml_data = yaml.load(yml_file, Loader=yaml.BaseLoader)
            if yml_data:
                flat_data = flatten_dict_or_list(yml_data)
                for key, value in flat_data.items():
                    parts = key.split('.')
                    new_parts = [part for part in parts if not part.isdigit()]
                    new_key = '.'.join(new_parts)
                    parsed_data.append({
                        'Repository': repository,
                        'CI File Path': ci_file_path,
                        'Local File Path': yml_path,
                        'CI Tool': tool_name,
                        'Item': new_key,
                        'Value': value
                    })

        except yaml.YAMLError as e:
            print(f"Error parsing {yml_path}: {e}")

    return parsed_data

folder_path = "../Data/ConfigFiles"
output_json_file = "../Data/config_items_values.json"

parsed_data = []

config_items_data = pd.read_csv('../Data/ci_config_files_dataset.csv')
filtered_repos = pd.read_csv('../Data/our_latest_repos.csv')
filtered_config_files = config_items_data[config_items_data['Repository'].isin(filtered_repos['Repository'])]
data = filtered_config_files

for _, group in data.iterrows():
    parsed_data.extend(parse_yml_files(group))

with open(output_json_file, 'w') as jsonfile:
    json.dump(parsed_data, jsonfile, indent=4)

09: Classifying directives based on CI/CD phase involves manual analysis. The result of this process is saved in the file path ../Data/refined_categorized_data.json

10: Link each directive to its category

In [ ]:
import pandas as pd

# Load the JSON files into Pandas DataFrames
config_items_values = pd.read_json('../Data/config_items_values.json')
refined_categorized_data = pd.read_json('../Data/refined_categorized_data.json')
repos_meta_data = pd.read_csv('../Data/new_merged_dataset.csv')

# Merge the first two DataFrames based on the common columns
merged_data = pd.merge(config_items_values, refined_categorized_data, left_on=['CI Tool', 'Item', 'Value'], right_on=['CI/CD Platform', 'Directive', 'Value'], how='left')

# Merge the merged_data DataFrame with the third DataFrame based on a common column
final_result = pd.merge(merged_data, repos_meta_data, left_on='Repository', right_on='App_Name', how='left')

# Save the final result to a new CSV file
final_result.to_json('../Data/final_result.json', orient='records')

11: Collect additional project's characteristics (commits, contributor, pull_requests)

In [ ]:
import csv
import time
import requests
from datetime import datetime, timedelta

# === Token and Request Setup ===

# Define a list of authorization headers
authorization_headers = [
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key2}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key3}"}
]

token_index = 0
last_request_time = {}

def makeRequest(api_url, params):
    global token_index
    num_tries = 0
    max_tries = 5

    while True:
        num_tries += 1
        if num_tries > max_tries:
            print(' -- Max retries reached. Exiting...')
            break

        available_token = None
        current_time = time.time()
        for i in range(len(authorization_headers)):
            if i not in last_request_time or (current_time - last_request_time[i] >= 1):
                available_token = i
                break

        if available_token is None:
            min_wait = min(last_request_time.values()) + 1 - current_time
            time.sleep(max(min_wait, 1))
            continue

        token_index = available_token
        headers = authorization_headers[token_index]
        last_request_time[token_index] = time.time()

        try:
            response = requests.get(api_url, headers=headers, params=params, timeout=30)
        except requests.exceptions.RequestException as e:
            print(f' -- Request error: {e}. Retrying in 60 seconds...')
            time.sleep(60)
            continue

        if response.status_code == 200:
            return response.json()
        elif response.status_code == 403 and 'rate limit' in response.text.lower():
            last_request_time[token_index] = time.time()
            continue
        elif response.status_code in {422, 401}:
            return None
        else:
            print(f' -- Status code {response.status_code}. Retrying... {api_url}')
            time.sleep(1)

# === Date-based Filtering ===

def convert_to_datetime(date_str):
    return datetime.strptime(date_str, '%Y-%m-%dT%H:%M:%SZ')

def get_commit_count(repo_full_name, start_date, end_date):
    url = f'https://api.github.com/repos/{repo_full_name}/commits'
    total_count = 0
    current_start = start_date
    step = timedelta(days=128)

    while current_start < end_date:
        current_end = min(current_start + step, end_date)
        print(f"Checking {current_start.date()} to {current_end.date()}")
        page = 1
        count_in_window = 0
        page_limit_hit = False

        while True:
            params = {
                'since': current_start.isoformat() + 'Z',
                'until': current_end.isoformat() + 'Z',
                'per_page': 100,
                'page': page
            }
            time.sleep(1)
            data = makeRequest(url, params)
            if not data:
                print(f" -- No commits on page {page}")
                break
            print(f" -- Commits page {page}: {len(data)}")
            count_in_window += len(data)
            if len(data) < 100:
                break
            page += 1
            if page > 10:
                if step == timedelta(days=1):
                    print(" -- Step reached 1 day and still too many commits. Aborting.")
                    return '<many>'
                print(" -- Hit page limit, reducing step size and retrying")
                step = step / 2
                page_limit_hit = True
                break

        if not page_limit_hit:
            total_count += count_in_window
            if step == timedelta(days=1):
                current_start = current_end + step
                step = timedelta(days=16)
            else:
                current_start = current_end 

    return total_count

def get_pr_count(repo_full_name, before_date):
    query = f'repo:{repo_full_name} is:pr created:<{before_date.date()}'
    url = 'https://api.github.com/search/issues'
    params = {'q': query, 'per_page': 1}
    data = makeRequest(url, params)
    return data['total_count'] if data and 'total_count' in data else 0

def get_contributor_count(repo_full_name):
    total_count = 0
    page = 1
    headers = {'Accept': 'application/vnd.github.v3+json', 'Authorization': f'Bearer {tokens[0]}'}

    while True:
        url = f'https://api.github.com/repos/{repo_full_name}/contributors'
        params = {'per_page': 100, 'page': page, 'anon': 'false'}
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        if response.status_code != 200 or not data:
            print(f"Error or empty response on page {page}: {response.status_code}")
            break
        print(f" -- Contributors page {page}: {len(data)}")
        total_count += len(data)
        if len(data) < 100:
            break
        page += 1

    return total_count

def get_repo_data(repo_full_name, start_date, before_date):
    commits = get_commit_count(repo_full_name, start_date, before_date)
    time.sleep(1)
    prs = get_pr_count(repo_full_name, before_date)
    time.sleep(1)
    contributors = get_contributor_count(repo_full_name)
    time.sleep(2)
    return commits, prs, contributors

def resolve_repo_redirect(owner, repo):
    url = f'https://api.github.com/repos/{owner}/{repo}'
    for token_id, headers in enumerate(authorization_headers):
        try:
            response = requests.get(url, headers=headers, allow_redirects=True, timeout=10)
            if response.status_code == 200:
                data = response.json()
                full_name = data.get('full_name')
                if full_name:
                    return full_name  # New or unchanged repo name
            elif response.status_code == 404:
                print(f" -- Repo not found: {owner}/{repo}")
                return None
        except Exception as e:
            print(f" -- Error resolving {owner}/{repo}: {e}")
    return None

# === Main Execution ===

input_file = '../Data/full_dataset.csv'
output_file = '../Data/android_repos_all_metrics.csv'
start_date = datetime(2008, 1, 1)
end_date = datetime(2023, 5, 31)

with open(input_file, 'r') as infile, open(output_file, 'a', newline='') as outfile:
    reader = csv.DictReader(infile)
    writer = csv.DictWriter(outfile, fieldnames=['Repository', 'Resolved_Repository', 'commits', 'pull_requests', 'contributors'])
    writer.writeheader()

    for row in reader:
        original_repo = row['Repository'].strip()
        if '/' not in original_repo:
            continue

        owner, repo_name = original_repo.split('/')
        resolved_repo = resolve_repo_redirect(owner, repo_name)

        if not resolved_repo:
            print(f"Skipping unresolved repo: {original_repo}")
            writer.writerow({
                'Repository': original_repo,
                'Resolved_Repository': '',
                'commits': 0,
                'pull_requests': 0,
                'contributors': 0
            })
            outfile.flush()
            continue
        
        print(f"Processing {original_repo} -> {resolved_repo} ...")
        try:
            commits, prs, contributors = get_repo_data(resolved_repo, start_date, end_date)
        except Exception as e:
            print(f" -- Error with {original_repo}: {e}")
            commits, prs, contributors = 'error', 'error', 'error'

        writer.writerow({
            'Repository': original_repo,
            'Resolved_Repository': resolved_repo,
            'commits': commits,
            'pull_requests': prs,
            'contributors': contributors
        })
        outfile.flush()
        print(f" --> {original_repo}: commits={commits}, PRs={prs}, contributors={contributors}")
        time.sleep(1)


12: Visualize project's characteristics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np

# Load data
df = pd.read_csv('../Data/android_repos_all_metrics.csv')

# Metrics to plot
metrics = ['Size', 'commits', 'pull_requests', 'Issues_Count', 'contributors', 'Repo_Stars', 'Repo_Forks']
labels = ['Repo Size (MB)', '# of Commits', '# of Pull Requests', '# of Issues', '# of Contributors', '# of Stars', '# of Forks']

# Y-axis tick formatter
def log_tick_formatter(x, pos):
    return f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'

formatter = FuncFormatter(log_tick_formatter)

# Create subplots
fig, axes = plt.subplots(nrows=1, ncols=7, figsize=(13, 4), constrained_layout=True)
axes = axes.flatten()

stats = []
for ax, metric, label in zip(axes, metrics, labels):
    if metric == 'Size':
        data = (df[metric] / 1024).dropna()
    else:
        data = df[metric].dropna()
    box = ax.boxplot(data, widths=0.4)

    ax.set_title(label, fontsize=12)

    # Log scale and formatting
    ax.set_yscale('log')
    ax.set_yticks([1, 10, 100, 1000, 10000, 100000])
    ax.yaxis.set_major_formatter(formatter)
    ax.yaxis.set_minor_locator(plt.NullLocator())  # <- disables minor ticks
    ax.set_xticks([])
    ax.set_ylabel('')
    ax.tick_params(axis='both', labelsize=11)

    # Only major gridlines
    ax.grid(True, which='major', axis='y', linestyle='--', linewidth=0.5)

    # Annotate median
    median = np.median(data)
    ax.text(0.95, median, f'   {int(median):,}', va='bottom', ha='center', fontsize=11)

    # Print statistics
    stats.append({
        'Metric': label,
        'Min': round(data.min(), 2),
        'Mean': int(data.mean()),
        'Median': int(data.median()),
        'Max': int(data.max())
    })

plt.savefig('boxplots_all_repo_characteristics.pdf')
plt.show()

# Create DataFrame and display
stats_df = pd.DataFrame(stats)
print(stats_df)
